# Notebook 08: Project Integration

**Time:** 35 minutes  
**Prerequisites:** Notebooks 02-07 complete  
**Goal:** Apply this week's tools to your capstone project and plan your data pipeline

This notebook will:
1. Review what you learned this week
2. Ask Claude to design a domain-specific data pipeline for your project
3. Run a mini pipeline on your project's data
4. Generate a project update document

In [1]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 08")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
Setup complete -- ready for Notebook 08


---

## Part 1: Week 3 Review

This week you learned:
- **Web scraping**: trafilatura (traditional) vs Crawl4AI (modern LLM-ready)
- **OCR**: Tesseract (baseline) -> Surya (layout-aware) -> Marker/Docling (2025)
- **ASR**: Whisper, faster-whisper, model size trade-offs
- **Data cleaning**: MinHash dedup, PII removal, DataTrove/FineWeb architecture
- **TTS**: edge-tts (free cloud), Kokoro (local, highest quality)
- **Voice agents**: ASR->LLM->TTS pipeline, Pipecat framework

In [2]:
# Load previous project definition (from HW1/HW2) or create a blank
prev_project = os.path.join(outputs_dir, 'my_project_update.md')
hw2_project = os.path.join(parent_dir, '..', 'Homework2-Submission', 'outputs', 'my_project_update.md')
hw1_project = os.path.join(parent_dir, '..', 'Homework1-Submission', 'outputs', 'my_project_definition.md')

project_context = ""
for path in [hw2_project, hw1_project]:
    if os.path.exists(path):
        with open(path, 'r') as f:
            project_context = f.read()
        print(f"Loaded project context from: {path}")
        print(f"Content preview: {project_context[:300]}...")
        break

if not project_context:
    print("No previous project definition found.")
    print("You'll define your project focus below.")

No previous project definition found.
You'll define your project focus below.


---

## Part 2: Data Pipeline Strategy

In [3]:
# TODO 1: Define your project's data strategy

print("=" * 65)
print("TODO 1: Project Data Pipeline Design")
print("=" * 65)
print()

project_description = "I just started a capstone project where I'm building a semiconductor focus application that helps engineers quickly find relevant information from technical documents, research papers, and web sources. The goal is to create a tool that can extract key insights, summarize findings, and provide voice-based interactions for users who are on the go."
project_domain = "Engineering / Semiconductors"

strategy_prompt = f"""I'm building this capstone project:
{project_description}

Domain: {project_domain}

Previous context:
{project_context[:500] if project_context else 'No previous context'}

This week I learned about:
- Web scraping (trafilatura, Crawl4AI)
- OCR (Tesseract, Marker, Docling)
- ASR (faster-whisper)
- Data cleaning (MinHash dedup, PII removal, quality filtering)
- TTS (edge-tts, Kokoro)
- Voice agents (Pipecat framework)

Design a data pipeline strategy for my project:
1. What data sources should I collect from? (web, PDFs, audio?)
2. Which extraction tools are best for my domain?
3. What cleaning steps are critical for my use case?
4. Should I incorporate voice capabilities? If so, how?
5. What's the estimated data volume and processing time?

Be specific and practical."""

start = time.time()
response = client.generate(
    prompt=strategy_prompt,
    system="You are a senior ML engineer helping design a data pipeline for a capstone project.",
    max_tokens=700,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")

todo1_reflection = """
[YOUR REFLECTION HERE]

- Which data sources are most relevant for your project?
- I think the most useful extraction tools for my domain will be web scraping for technical documents and research papers like IEEE, and OCR for any scanned PDFs. ASR might be less critical unless I want to include audio sources.
- Which tools from this week will you actually use in your capstone?
- I think I'll primarily use trafilatura for web scraping, Tesseract for OCR, and MinHash for deduplication. I might experiment with faster-whisper for ASR if I find relevant audio content.
- What's the most challenging data quality issue you expect to face?
- I anticipate that a major data quality issue will be dealing with noisy and unstructured data from web sources. Ensuring that I can extract clean, relevant information without too much irrelevant content will be a challenge.
"""

print()
print(todo1_reflection)

TODO 1: Project Data Pipeline Design

Response in 15.1s
Model: claude-sonnet-4-6
Tokens: 293 in, 700 out
Stop reason: max_tokens
# Data Pipeline Strategy: Semiconductor Intelligence Tool

## Executive Summary

You're building a **Retrieval-Augmented Generation (RAG) system** specialized for semiconductor engineering. Your pipeline needs to handle highly technical content with precise terminology, equations, and structured data (tables, figures, specs). Here's a concrete, buildable strategy.

---

## 1. Data Sources

### Primary Sources (High Priority)

```
Semiconductor Knowledge Base
├── Technical PDFs
│   ├── Datasheets (Texas Instruments, Infineon, STMicro portals)
│   ├── Application Notes (vendor websites)
│   ├── IEEE Xplore papers (if licensed) or arXiv EE section
│   └── JEDEC/IEC standards documents
│
├── Web Sources
│   ├── Electronics Stack Exchange (Q&A gold mine)
│   ├── SemiWiki.com (industry blog/forum)
│   ├── EETimes.com (news + technical articles)
│   ├── Vendor docum

---

## Part 3: Mini Pipeline Demo

In [8]:
# TODO 2: Run a mini data pipeline for your project
#
# Use at least 2 tools from this week to collect and clean
# a small sample of data relevant to your project.

print("=" * 65)
print("TODO 2: Mini Pipeline for Your Project")
print("=" * 65)
print()

# Example: scrape papers + clean them
from src.scraping_utils import scrape_arxiv_abstracts
from src.data_pipeline import run_cleaning_pipeline

# Step 1: Collect data
my_topic = "Semiconductor manufacturing"
papers = scrape_arxiv_abstracts(topic=my_topic, max_results=5)

# Step 2: Clean the data
raw_texts = [p['abstract'] for p in papers]
pipeline_result = run_cleaning_pipeline(
    texts=raw_texts,
    target_lang="en",
    save_path=os.path.join(outputs_dir, 'my_project_data.json'),
)

print(f"\nCollected and cleaned {len(pipeline_result['cleaned_texts'])} documents for your project.")

todo2_reflection = """
[YOUR REFLECTION HERE]

- What data did you collect and how did you clean it?
- I collected abstracts of research papers related to semiconductor manufacturing from arXiv. I then ran a cleaning pipeline that included language detection, deduplication using MinHash, and quality filtering to remove any low-quality or irrelevant abstracts.
- Were any documents removed by the pipeline? Why?
- No documents were removed during the quality filtering step due to low relevance or poor text quality.
- How would you scale this to a full dataset for your project?
- To scale this to a full dataset, I would set up an automated pipeline that continuously scrapes new papers from relevant sources like arXiv, IEEE, and other research databases. I would also implement more robust error handling and monitoring to ensure the pipeline runs smoothly. Additionally, I might consider parallelizing the scraping and cleaning processes to handle larger volumes of data more efficiently.
"""

print()
print(todo2_reflection)

TODO 2: Mini Pipeline for Your Project

Scraping arXiv: 'Semiconductor manufacturing' (max 5 papers)
  Attempt 1/3...

  [1] The Structural Case for the Eco-Civilization Paradigm...
      Authors: Lei Zhu, William Zhu
      Abstract: This paper establishes a quantitative and structural framework for civilizational continuity under rapid, non-linear eco...

  [2] Overcoming contact resistance at metal-2D semiconductor interfaces: atomically c...
      Authors: Rafal Dunal, Maxime Le Ster, Iaroslav Lutsyk...
      Abstract: The application of two-dimensional (2D) semiconductors, such as monolayer MoS2, is limited by the high contact resistanc...

  [3] Origin of the Temperature-Induced Gap Bowing of Formamidinium-Methylammonium Lea...
      Authors: Kai Xu, Adrián Francisco-López, Bethan L. Charles...
      Abstract: A thorough understanding of the temperature dependence of semiconductor band gaps is essential for optimizing optoelectr...

  [4] BatteryMFormer: Multi-level Learning for B

---

## Part 4: Generate Project Update

In [9]:
# Generate project update document
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[Not completed]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[Not completed]'

project_update = f"""# Week 3 Project Update: Pretraining Data & Voice Agents

## Project Description

{project_description}

## Data Pipeline Strategy

{response['content'] if 'error' not in response else 'Error generating strategy'}

## Mini Pipeline Results

- Documents collected: {len(raw_texts) if 'raw_texts' in dir() else 'N/A'}
- Documents after cleaning: {len(pipeline_result['cleaned_texts']) if 'pipeline_result' in dir() else 'N/A'}

## Reflections

### Data Strategy
{_todo1}

### Pipeline Execution
{_todo2}

## Tools I Plan to Use

Trafilatura, Tesseract, MinHash, and possibly faster-whisper for ASR.

## Next Steps

I will continue refining my data pipeline, potentially adding more sources and improving the cleaning steps. I also want to start exploring how to integrate voice capabilities into my project, perhaps by prototyping a simple voice agent using Pipecat.
"""

update_path = os.path.join(outputs_dir, 'my_project_update.md')
with open(update_path, 'w') as f:
    f.write(project_update)

print(f"Project update saved: {update_path}")

Project update saved: ..\outputs\my_project_update.md


In [10]:
# Final reflection
full_reflection = f"""
### Data Pipeline Strategy

{_todo1}

---

### Mini Pipeline Results

{_todo2}
"""

reflection_file = append_to_reflection(
    notebook="08",
    section_title="Project Integration",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ..\outputs\homework_reflection.md

API COST REPORT
Total API calls:     1
Total input tokens:  293
Total output tokens: 700
Total cost:          $0.0114

Last 1 calls:
  1. [00:27:11] sonnet -- 293in/700out -- $0.0114


---

## Week 3 Complete!

### Submission Checklist

- [ ] All notebooks (00-08) executed with TODOs filled in
- [ ] `outputs/homework_reflection.md` -- your main deliverable (70%)
- [ ] `outputs/my_project_update.md` -- project integration (20%)
- [ ] `outputs/arxiv_papers.json` -- scraped paper data
- [ ] `outputs/cleaned_data.json` -- cleaned pipeline output
- [ ] Audio files in outputs/ -- TTS demonstrations

### What's Next?

**Week 4:** Retrieval-Augmented Generation (RAG) -- combining LLMs with external knowledge, vector databases, and LangChain.